# Quantitative Feature Engineering

This notebook demonstrates leakage-aware feature construction from validated historical OHLCV data. It does not create prediction targets or train a model.

## Objective

Generate deterministic lag, momentum, trend, volatility, volume, RSI, MACD, and ATR features using only information available at or before each timestamp.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'ml').exists():
    project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml.data.ingestion import MarketDataIngestionService
from ml.data.yahoo import YahooFinanceProvider
from ml.features.pipeline import build_features
from ml.features.validation import validate_features
from ml.visualization.market_plots import plot_close_history, plot_rolling_volatility

## Load Validated AAPL OHLCV

Use the existing local raw CSV when available, otherwise retrieve data through the existing ingestion service. This cell is intentionally unexecuted in the committed notebook.

In [ ]:
symbol = 'AAPL'
raw_path = project_root / 'data' / 'raw' / f'{symbol}.csv'
if raw_path.exists():
    ohlcv = pd.read_csv(raw_path, parse_dates=['date'])
else:
    service = MarketDataIngestionService(YahooFinanceProvider())
    ohlcv = service.ingest(symbol, '2020-01-01', '2026-01-01')
ohlcv.head()

## Generate and Inspect Features

In [ ]:
features = build_features(ohlcv)
validate_features(features)
print(features.columns.tolist())
features.head()

## Warm-up NaNs

Lagged, rolling, and indicator features naturally begin with missing values while their historical windows are being established. They are not backfilled from future observations.

In [ ]:
features.isna().sum().sort_values(ascending=False).head(15)

## Trend Features

In [ ]:
trend_view = features.set_index('date')[['close', 'sma_20', 'ema_12', 'ema_26']]
trend_view.plot(title='Closing price and trailing trend features', ylabel='Price')

## Volatility and Momentum

In [ ]:
volatility_view = features.set_index('date')['volatility_20']
figure, axes = plot_rolling_volatility(volatility_view)
figure.show()
features[['date', 'momentum_5', 'momentum_20']].tail()

## Volume Features

In [ ]:
features[['date', 'volume_change', 'volume_lag_1', 'volume_sma_20', 'relative_volume_20']].tail()

## RSI, MACD, and ATR

In [ ]:
features[['date', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14']].tail()

## Why Leakage Matters

Every feature is trailing or point-in-time: lags use positive shifts, rolling windows are not centered, and no feature is backfilled. Appending future OHLCV rows must not change values in the historical prefix; this property is covered by automated tests.

No prediction target exists yet. This phase creates explanatory inputs only, not target returns, target direction, signals, or model data splits.